<a href="https://colab.research.google.com/github/viki2408/SQL-Query-Generator/blob/main/SQL_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 64.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from transformers import pipeline
pipe = pipeline("text-generation",model="ibm-granite/granite-3.3-2b-instruct")
messages=[{
    "role" : "users","content": "who are you",
}]
pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'users', 'content': 'who are you'},
   {'role': 'assistant',
    'content': 'I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses.'}]}]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")

messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.<|end_of_text|>


In [3]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 17.5 MB/s eta 0:00:00


In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests sqlite3


In [4]:
import gradio as gr
from datetime import datetime
from pathlib import Path
import json
import sqlite3
import re
import urllib.parse

from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

# =========================================================
# DATABASE SCHEMA
# =========================================================

SCHEMA = {
    "users": {
        "id": "INTEGER PRIMARY KEY",
        "name": "TEXT",
        "email": "TEXT",
        "signup_date": "DATE",
        "age": "INTEGER",
        "country": "TEXT",
        "status": "TEXT",
    },
    "orders": {
        "id": "INTEGER PRIMARY KEY",
        "user_id": "INTEGER",
        "product_name": "TEXT",
        "amount": "REAL",
        "order_date": "DATE",
        "status": "TEXT",
    },
    "products": {
        "id": "INTEGER PRIMARY KEY",
        "name": "TEXT",
        "price": "REAL",
        "category": "TEXT",
        "stock": "INTEGER",
    },
    "transactions": {
        "id": "INTEGER PRIMARY KEY",
        "user_id": "INTEGER",
        "amount": "REAL",
        "date": "DATE",
        "type": "TEXT",
    },
}

# =========================================================
# SECURITY
# =========================================================

DANGEROUS_KEYWORDS = [
    "DROP",
    "DELETE",
    "ALTER",
    "TRUNCATE",
    "INSERT",
    "UPDATE",
    "EXEC",
    "EXECUTE",
]

DANGEROUS_CHARS = [";", "--", "/*", "*/", "xp_", "sp_"]


def is_safe(text):
    text_upper = text.upper()
    for keyword in DANGEROUS_KEYWORDS:
        if keyword in text_upper:
            return False
    for char in DANGEROUS_CHARS:
        if char in text:
            return False
    return True


# =========================================================
# TABLE DETECTION
# =========================================================

def get_table_name(query_text):
    query_lower = query_text.lower()

    keyword_map = {
        "users": ["user", "users", "customer", "customers", "account"],
        "orders": ["order", "orders", "purchase", "buy"],
        "products": ["product", "products", "item", "items"],
        "transactions": ["transaction", "transactions", "payment"],
    }

    for table, keywords in keyword_map.items():
        for kw in keywords:
            if kw in query_lower:
                return table

    return "users"


def get_column_names(table, query_text):
    columns = list(SCHEMA[table].keys())
    query_lower = query_text.lower()

    selected_cols = []
    for col in columns:
        if col.lower() in query_lower:
            selected_cols.append(col)

    if not selected_cols:
        return ["*"]

    return list(dict.fromkeys(selected_cols))


# =========================================================
# WHERE CONDITIONS
# =========================================================

def extract_conditions(query_text):
    query_lower = query_text.lower()
    conditions = []

    # last month
    if "last month" in query_lower:
        conditions.append("DATE(signup_date) >= DATE('now','-1 month')")

    # last year
    if "last year" in query_lower or "this year" in query_lower:
        conditions.append("DATE(signup_date) >= DATE('now','-1 year')")

    # today
    if "today" in query_lower:
        conditions.append("DATE(signup_date) = DATE('now')")

    # active/inactive
    if "active" in query_lower:
        conditions.append("status = 'active'")
    elif "inactive" in query_lower:
        conditions.append("status = 'inactive'")

    # amount filtering
    match = re.search(r"(greater than|more than|above)\s*(\d+)", query_lower)
    if match:
        conditions.append(f"amount > {match.group(2)}")

    match = re.search(r"(less than|below)\s*(\d+)", query_lower)
    if match:
        conditions.append(f"amount < {match.group(2)}")

    return conditions


# =========================================================
# SQL GENERATOR
# =========================================================

def generate_sql(query_text):
    if not is_safe(query_text):
        return None, "❌ UNSAFE: Query contains dangerous SQL keywords!"

    table = get_table_name(query_text)
    columns = get_column_names(table, query_text)
    columns_str = ", ".join(columns)

    sql = f"SELECT {columns_str} FROM {table}"

    conditions = extract_conditions(query_text)
    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    if "all" not in query_text.lower():
        sql += " LIMIT 10"

    return sql, f"✅ Generated from table '{table}'"


# =========================================================
# CHAT PROCESS
# =========================================================

query_history = []


def process_query(user_query, chat_history):
    if not user_query.strip():
        return chat_history

    chat_history.append((user_query, "Processing..."))

    sql_query, explanation = generate_sql(user_query)

    if sql_query is None:
        response = f"❌ ERROR\n\n{explanation}"
    else:
        response = f"✅ SQL GENERATED\n\n```sql\n{sql_query}\n```\n\n{explanation}"
        query_history.append(
            {
                "user": user_query,
                "sql": sql_query,
                "timestamp": datetime.now().isoformat(),
            }
        )

    chat_history[-1] = (user_query, response)
    return chat_history


# =========================================================
# EXPORT PDF
# =========================================================

def export_pdf():
    if not query_history:
        return "❌ No queries to export", None

    filename = f"sql_queries_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"

    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("SQL Query Report", styles["Heading1"]))
    story.append(Spacer(1, 0.25 * inch))

    for q in query_history[-5:]:
        story.append(Paragraph(f"<b>User:</b> {q['user']}", styles["Normal"]))
        story.append(Paragraph(f"<b>SQL:</b> {q['sql']}", styles["Normal"]))
        story.append(Spacer(1, 0.2 * inch))

    doc.build(story)

    return f"✅ PDF Downloaded: {filename}", filename


# =========================================================
# WHATSAPP LINK
# =========================================================

def generate_whatsapp_link():
    if not query_history:
        return "❌ No queries to share"

    message = "📊 SQL Queries Generated\n\n"

    for i, q in enumerate(query_history[-5:], 1):
        message += f"*Query {i}:* {q['user']}\n"
        message += f"{q['sql']}\n\n"

    encoded = urllib.parse.quote(message)
    link = f"https://wa.me/?text={encoded}"

    return f"✅ WhatsApp Link Created\n\n🔗 {link}"


# =========================================================
# UI
# =========================================================

with gr.Blocks() as demo:
    gr.Markdown(
        "## 🚀 Natural Language → SQLite SQL\n"
        "Convert plain English requests into safe SQL queries!"
    )

    chatbot = gr.Chatbot(height=450)

    query_input = gr.Textbox(
        label="Natural Language Request",
        placeholder="e.g., Show me all users who signed up last month",
    )

    with gr.Row():
        submit_btn = gr.Button("Generate SQL")
        pdf_btn = gr.Button("Download PDF")
        wa_btn = gr.Button("Share WhatsApp")

    file_output = gr.File()
    status = gr.Markdown()

    submit_btn.click(process_query, [query_input, chatbot], chatbot)
    pdf_btn.click(export_pdf, outputs=[status, file_output])
    wa_btn.click(generate_whatsapp_link, outputs=status)

# =========================================================
# LAUNCH
# =========================================================

demo.launch(share=True, show_error=True, show_api=False)

/tmp/ipykernel_287/3337111621.py:275: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450)
/tmp/ipykernel_287/3337111621.py:275: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=450)
/tmp/ipykernel_287/3337111621.py:298: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  demo.launch(share=True, show_error=True, show_api=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://64ea17901f1486d189.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
